# Auditoría del experimento — sentiment140

Este notebook reconstruye, consultando el Tracking Server en tiempo real, el protocolo experimental, los runs registrados, las contribuciones individuales y la trazabilidad del modelo desplegado. Sirve de apoyo para la sustentación (Sección 8 de la guía) y **no** forma parte del contrato de caja negra: la evidencia oficial vive en MLflow, este notebook solo la muestra de forma legible.

Antes de ejecutar, confirma que `MLFLOW_TRACKING_URI` apunte al Tracking Server correcto.

In [26]:
import os
import mlflow
import pandas as pd

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://54.87.148.165:5000")
EXPERIMENT_NAME = "nlp-lab2-sentiment140"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = mlflow.MlflowClient()

exp = client.get_experiment_by_name(EXPERIMENT_NAME)
assert exp is not None, f"No existe el experimento {EXPERIMENT_NAME}"
print(f"Tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {exp.name}  (experiment_id={exp.experiment_id})")

Tracking URI: http://54.87.148.165:5000
Experiment: nlp-lab2-sentiment140  (experiment_id=1)


## 1. Todos los runs presentados

In [27]:
runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="",
    max_results=1000,
)

rows = []
for r in runs:
    tags = r.data.tags
    run_type = tags.get("lab_run_type")
    if run_type not in ("protocol", "experiment", "final"):
        continue  # runs exploratorios: se ignoran (A.1)
    rows.append({
        "run_id": r.info.run_id,
        "status": r.info.status,
        "run_type": run_type,
        "lab_experiment_id": tags.get("lab_experiment_id", ""),
        "lab_stage": tags.get("lab_stage", ""),
        "lab_member_id": tags.get("lab_member_id", ""),
        "macro_f1_mean": r.data.metrics.get("macro_f1_mean"),
        "macro_f1_std": r.data.metrics.get("macro_f1_std"),
        "test_macro_f1": r.data.metrics.get("test_macro_f1"),
        "macro_f1_delta": r.data.metrics.get("macro_f1_delta"),
    })

runs_df = pd.DataFrame(rows).sort_values("run_id").reset_index(drop=True)
print(f"Total de runs presentados: {len(runs_df)}")
runs_df

Total de runs presentados: 19


,run_id,status,run_type,lab_experiment_id,lab_stage,lab_member_id,macro_f1_mean,macro_f1_std,test_macro_f1,macro_f1_delta
0,1aa53612fb874100a6a84fc5286f942a,FINISHED,experiment,C_LOGREG,classifier,carlos.cardona02,0.800957,0.000095,NaN,NaN
1,27d1e7851daa49189984963cf639a427,FINISHED,experiment,ABLATION,ablation,carlos.cardona02,0.800028,0.000167,NaN,0.000929
2,3d768e180fde4a1d835d589ad7f498c0,FINISHED,experiment,P_LEMMA,preprocessing,carlos.cardona02,0.779270,0.000799,NaN,NaN
3,41ce19e6e4954a9a9cd0d8c9ee3a48f1,FINISHED,protocol,,,,NaN,NaN,NaN,NaN
4,551674e64aba4e0e9d91d66be6086177,FINISHED,experiment,P_ELONGATION,preprocessing,carlos.cardona02,0.784145,0.000603,NaN,NaN
5,59e2947f516149d7850a65328d2d58bd,FINISHED,experiment,C_LINEAR_SVM,classifier,carlos.cardona02,0.799271,0.001075,NaN,NaN
6,6efe2e4348264f1c8141f976bae6dea0,FINISHED,experiment,R_TFIDF_UNI_BI,representation,carlos.cardona02,0.800957,0.000095,NaN,NaN
7,6ffcd33a2c3747748410b880a58f60d5,FINISHED,experiment,C_SGD,classifier,carlos.cardona02,0.780881,0.000613,NaN,NaN
8,8ec735c9619e4c8aa7011097f02f7b3a,FINISHED,experiment,B0,baseline,carlos.cardona02,0.783072,0.001158,NaN,NaN
9,99a48bb2be234540a7a6b92ebc12e739,FINISHED,experiment,R_TFIDF_UNI,representation,carlos.cardona02,0.789977,0.000250,NaN,NaN


## 2. Run de protocolo

In [28]:
protocol_runs = [r for r in runs if r.data.tags.get("lab_run_type") == "protocol"]
assert len(protocol_runs) == 1, f"Se esperaba exactamente 1 run de protocolo, hay {len(protocol_runs)}"
protocol_run = protocol_runs[0]

print("protocol_run_id:", protocol_run.info.run_id)
for k, v in protocol_run.data.params.items():
    print(f"  {k} = {v}")

artifacts = client.list_artifacts(protocol_run.info.run_id, "protocol")
print("\nArtefactos:")
for a in artifacts:
    print(" -", a.path)

protocol_run_id: 41ce19e6e4954a9a9cd0d8c9ee3a48f1
  dataset_id = adilbekovich/Sentiment140Twitter
  dataset_revision = b6037e127257d95b9b23d31f78b264b9ebe697fd
  sampling_strategy = stratified
  sample_size = 200000
  random_seed = 42
  cv_strategy = StratifiedKFold
  cv_folds = 3
  cv_shuffle = True

Artefactos:
 - protocol/members.csv
 - protocol/partitions.csv


In [29]:
# Verificación de protocol/partitions.csv: 200.000 filas, columnas index,fold
part_path = mlflow.artifacts.download_artifacts(
    run_id=protocol_run.info.run_id, artifact_path="protocol/partitions.csv"
)
partitions_df = pd.read_csv(part_path)
print("Filas:", len(partitions_df))
print("Columnas:", list(partitions_df.columns))
print("Folds únicos:", sorted(partitions_df["fold"].unique().tolist()))
print("Distribución por fold:")
print(partitions_df["fold"].value_counts().sort_index())

Filas: 200000
Columnas: ['index', 'fold']
Folds únicos: [0, 1, 2]
Distribución por fold:
fold
0    66667
1    66667
2    66666
Name: count, dtype: int64


In [30]:
members_path = mlflow.artifacts.download_artifacts(
    run_id=protocol_run.info.run_id, artifact_path="protocol/members.csv"
)
members_df = pd.read_csv(members_path)
members_df

,member_id,notebook_arn
0,carlos.cardona02,arn:aws:sagemaker:us-east-1:868565605844:noteb...


## 3. Comparaciones obligatorias por etapa

In [31]:
STAGE_ORDER = ["reference", "baseline", "preprocessing", "representation", "classifier", "ablation"]
exp_df = runs_df[runs_df["run_type"] == "experiment"].copy()
exp_df["lab_stage"] = pd.Categorical(exp_df["lab_stage"], categories=STAGE_ORDER, ordered=True)
exp_df = exp_df.sort_values(["lab_stage", "lab_experiment_id"])

for stage in STAGE_ORDER:
    subset = exp_df[exp_df["lab_stage"] == stage]
    if len(subset) == 0:
        continue
    print(f"\n=== {stage} ===")
    cols = ["lab_experiment_id", "macro_f1_mean", "macro_f1_std", "macro_f1_delta"]
    display(subset[cols].reset_index(drop=True))


=== reference ===


,lab_experiment_id,macro_f1_mean,macro_f1_std,macro_f1_delta
0,T0,0.333376,0.000002,NaN



=== baseline ===


,lab_experiment_id,macro_f1_mean,macro_f1_std,macro_f1_delta
0,B0,0.783072,0.001158,NaN



=== preprocessing ===


...

,lab_experiment_id,macro_f1_mean,macro_f1_std,macro_f1_delta
0,R_BOW,0.784145,0.000603,NaN
1,R_SPACY,0.688145,0.000880,NaN
2,R_TFIDF_UNI,0.789977,0.000250,NaN
3,R_TFIDF_UNI_BI,0.800957,0.000095,NaN



=== classifier ===


,lab_experiment_id,macro_f1_mean,macro_f1_std,macro_f1_delta
0,C_LINEAR_SVM,0.799271,0.001075,NaN
1,C_LOGREG,0.800957,0.000095,NaN
2,C_SGD,0.780881,0.000613,NaN



=== ablation ===


,lab_experiment_id,macro_f1_mean,macro_f1_std,macro_f1_delta
0,ABLATION,0.800028,0.000167,0.000929
1,ABLATION,0.784145,0.000603,0.016812


## 4. Contribuciones individuales

In [32]:
import requests

API_BASE = os.getenv("API_BASE_URL", "http://54.87.148.165:8000")
resp = requests.get(f"{API_BASE}/audit/contributions", timeout=60)
resp.raise_for_status()
contrib = resp.json()

for m in contrib["members"]:
    ok = m["valid_configurations"] >= 3 and len(m["stages"]) >= 2
    print(f"{m['member_id']}: {m['valid_configurations']} configuraciones válidas, "
          f"etapas={m['stages']}  ->  {'CUMPLE' if ok else 'NO CUMPLE'} el mínimo")

print("\nRuns inválidos:", contrib["invalid_run_ids"])
print("Runs no atribuibles:", contrib["unattributed_run_ids"])

carlos.cardona02: 13 configuraciones válidas, etapas=['ablation', 'classifier', 'preprocessing', 'representation']  ->  CUMPLE el mínimo

Runs inválidos: ['b1c54a54962e4c03a20c4d9b115344ca']
Runs no atribuibles: []


## 5. Modelo final y trazabilidad

In [33]:
resp = requests.get(f"{API_BASE}/audit/model", timeout=60)
resp.raise_for_status()
model_info = resp.json()

print("model_name:", model_info["model_name"])
print("alias:", model_info["alias"])
print("version:", model_info["version"])
print("run_id (final):", model_info["run_id"])
print("protocol_run_id:", model_info["protocol_run_id"])
print("selected_experiment_run_id:", model_info["selected_experiment_run_id"])
print("training_size:", model_info["training_size"])
print("test_macro_f1:", model_info["test_macro_f1"])
print("\nconfiguration:")
import json
print(json.dumps(model_info["configuration"], indent=2, ensure_ascii=False))

model_name: sentiment140
alias: champion
version: 2
run_id (final): d92e32cb936542ebbf8afc72df402542
protocol_run_id: 41ce19e6e4954a9a9cd0d8c9ee3a48f1
selected_experiment_run_id: 1aa53612fb874100a6a84fc5286f942a
training_size: 1360000
test_macro_f1: 0.8262600946403855

configuration:
{
  "preprocessing": {
    "lowercase": true,
    "url": "token:url",
    "mention": "token:user",
    "whitespace": "normalize",
    "stopwords": "keep",
    "negators": [],
    "lemmatize": false,
    "elongation": "normalize",
    "elongation_spec": "reduce_repeated_chars_to_2",
    "emoji": "keep",
    "emoji_spec": null,
    "resources": {},
    "additional": {}
  },
  "representation": {
    "type": "tfidf",
    "ngram_range": [
      1,
      2
    ],
    "library": "scikit-learn",
    "library_version": "1.7.2",
    "spacy_model": null,
    "spacy_model_version": null,
    "parameters": {}
  },
  "classifier": {
    "type": "logistic_regression",
    "library": "scikit-learn",
    "library_version"

In [34]:
# Verificación de trazabilidad cruzada: /health, /audit/model y /api/v1/predict
# deben identificar el mismo run final (A.5)
health = requests.get(f"{API_BASE}/health", timeout=60).json()
predict = requests.post(
    f"{API_BASE}/api/v1/predict", json={"text": "this is a great example"}, timeout=15
).json()

print("health.model_run_id   =", health["model_run_id"])
print("audit/model.run_id    =", model_info["run_id"])
print("predict.model_run_id  =", predict["model_run_id"])

same = health["model_run_id"] == model_info["run_id"] == predict["model_run_id"]
print("\n¿Los tres coinciden?", "SÍ" if same else "NO -- revisar")

health.model_run_id   = d92e32cb936542ebbf8afc72df402542
audit/model.run_id    = d92e32cb936542ebbf8afc72df402542
predict.model_run_id  = d92e32cb936542ebbf8afc72df402542

¿Los tres coinciden? SÍ


## 6. Análisis de errores (resumen)

In [35]:
final_run_id = model_info["run_id"]
err_csv_path = mlflow.artifacts.download_artifacts(
    run_id=final_run_id, artifact_path="reports/error_analysis.csv"
)
err_df = pd.read_csv(err_csv_path)
print(f"Casos analizados: {len(err_df)}")
print("\nFrecuencia por categoría:")
print(err_df["category"].value_counts())
err_df

Casos analizados: 20

Frecuencia por categoría:
category
other              6
elongation         6
negation           3
informal           2
hashtag            1
contrast           1
intensification    1
Name: count, dtype: int64


,index,text,true_label,predicted_label,category
0,21582,is the only 1 that works around here,negative,positive,other
1,22381,@smandythk hey! mandy the wake was sooooo sad ...,negative,positive,elongation
2,23637,HSBC's website is broken one time more... To ...,negative,positive,hashtag
3,42954,@dinno and who knows u may find a new branch.....,positive,negative,informal
4,49229,I need my handy dandy shooting hat.. I think ...,negative,positive,other
5,88675,"There is not enough hours in a day for me, My ...",positive,negative,negation
6,96234,this just in: my bank account has reached an a...,positive,negative,other
7,104146,"Fun times last nite, but payin 4 it 2day... Co...",negative,positive,elongation
8,105425,off to geometry &amp; earth science,negative,positive,other
9,108165,is amazed at all the crap found while cleaning...,positive,negative,other


## 7. Resumen final

Esta celda imprime un resumen ejecutivo de todo lo verificado arriba, útil como referencia rápida durante la sustentación.

In [36]:
print("=" * 60)
print("RESUMEN DE AUDITORÍA")
print("=" * 60)
print(f"Experimento MLflow:        {EXPERIMENT_NAME}")
print(f"Runs presentados:          {len(runs_df)}")
print(f"Run de protocolo:          {protocol_run.info.run_id}")
print(f"Modelo final (champion):   {model_info['run_id']}  (versión {model_info['version']})")
print(f"Configuración final:       preprocessing.elongation=normalize, "
      f"representation=tfidf[1,2], classifier=logistic_regression")
print(f"test_macro_f1:             {model_info['test_macro_f1']:.4f}")
print(f"Trazabilidad /predict-/health-/audit/model: {'OK' if same else 'REVISAR'}")
print(f"Casos de análisis de errores: {len(err_df)}")
print("=" * 60)

RESUMEN DE AUDITORÍA
Experimento MLflow:        nlp-lab2-sentiment140
Runs presentados:          19
Run de protocolo:          41ce19e6e4954a9a9cd0d8c9ee3a48f1
Modelo final (champion):   d92e32cb936542ebbf8afc72df402542  (versión 2)
Configuración final:       preprocessing.elongation=normalize, representation=tfidf[1,2], classifier=logistic_regression
test_macro_f1:             0.8263
Trazabilidad /predict-/health-/audit/model: OK
Casos de análisis de errores: 20
